Hello Gentlemen, 

This JupyterNotebook is here to help us figure out if we can reliably look at a raman spectrum and determine how the reaction has progressed until now


Step 1: loading all the spectra and performing 1) baselining, 2) smoothing 3) normalisation

In [ ]:
import pandas as pd
from lamas.alpaca import Alpaca
from lamas.vicuna import Vicuna
from lamas.logger import Logger
from typing import Tuple
import matplotlib.pyplot as plt
import os


def plotting_function(x, y, label=None, fig=None):
    """
    Plots data onto an existing figure or creates a new figure if none is provided.

    :param fig: Optional; matplotlib Figure object. If None, creates a new figure.
    :param data: The data to plot; expects a dictionary with 'x' and 'y' keys.
    :param label: Optional; legend label for the data being plotted.
    :return: matplotlib Figure object with the plotted data.
    """

    if fig is None:
        fig, ax = plt.subplots()
    else:
        ax = fig.axes[0]

    ax.plot(x, y, label=label)

    if label is not None:
        ax.legend()

    return fig


log = Logger
log.start_logging_thread(platform="lamas", use_console=True)
# log = None
files = {
    # "250_000": "data/20241219/ES125-hd250_20241219_161004_063.csv",
    # "160_080": "data/20241219/ES125-160_20241220_090900_900.csv",
    # "080_160": "data/20241219/ES125-080_20241220_100422_277.csv",
    # "000_250": "data/20241219/ES125-hd000_20241219_163743_154.csv",
    # 'benzophenone': 'data/Benzophenone_20240215_013221.493.csv'
    # "12A": "data/Crudes/12A_crude_20250122_080853_665.csv"
}
path = "data/test_isotope_data_2/"
path_1 = "data/20250201/"
path_2 = "data/SPES44/FinalData/"
path_3 = "data/Scale_UP/"

files = {file: os.path.join(path, file) for file in os.listdir(path) if file.endswith(".csv")}
files.update({file: os.path.join(path_1, file) for file in os.listdir(path_1) if file.endswith(".csv")})
files_2 = {file: os.path.join(path_2, file) for file in os.listdir(path_2) if file.endswith(".csv")}
files_3 = {file: os.path.join(path_3, file) for file in os.listdir(path_3) if file.endswith(".csv")}

# for name, file in files.items():
#     print(name, file)
spectra = {}
fig = None
for name, file in files.items():
    # print(name, file)
    if "III" in name or "IV" in name:
        continue
    spectrum = Alpaca(logger=log)
    df = pd.read_csv(file, index_col=0)
    # spectrum.load_data(file, from_filetype = "ramaberry")
    spectrum.load_data(df, from_filetype="ramaberry")


    spectrum.crop_data(bounds=(1000, 3500))
    spectrum.baseline(method="cholesky", lam=10 ** 6, p=0.03, n_iter=20)
    spectrum.normalise_to(to=1000, range_vals=(2900, 2970))
    spectrum.crop_data(bounds=(1520, 2000))
    # # spectrum.denoise(method = "savgol", window_frac = 0.01)
    # # spectrum.denoise(method="wavelet", level=1)
    spectrum.denoise(method = "median", window = 5)
    # # spectrum.denoise(method='fft', threshold=0.6)
    # # spectrum.normalise_to(to=1000, range_vals=(1580, 1620))
    spectrum.crop_data(bounds=(1620, 1750))
    spectrum.baseline(method = "als", lam = 10**6, p=0.03, n_iter = 20)

    fig = plotting_function(spectrum.data[spectrum.x_col], spectrum.data[spectrum.y_cols[0]],# label=name.split("_")[0],
                            fig=fig)
    spectra[name.split("_")[0]] = spectrum

fig.axes[0].vlines(1685, -20,100)
fig.axes[0].set_ylim(-20, 40)
# plt.savefig('spectra.png', dpi = 1000)
plt.show()
print(spectra.keys())

fig_2 = None
spectra_2 = {}
for name, file in files_2.items():
    # if "III" in name or "IV" in name:
    #     continue
    # spectrum = Alpaca(logger=log)
    # df = pd.read_csv(file, index_col=0 )
    # # spectrum.load_data(file, from_filetype = "ramaberry")
    # spectrum.load_data(df, from_filetype="ramaberry")
    #
    # spectrum.crop_data(bounds=(1000, 3500))
    # spectrum.baseline(method="cholesky", lam=10 ** 6, p=0.03, n_iter=20)
    # spectrum.normalise_to(to=1000, range_vals=(2900, 2970))
    # spectrum.crop_data(bounds=(1520, 2000))
    # # # spectrum.denoise(method = "savgol", window_frac = 0.01)
    # # # spectrum.denoise(method="wavelet", level=1)
    # spectrum.denoise(method = "median", window = 5)
    # # # spectrum.denoise(method='fft', threshold=0.6)
    # # # spectrum.normalise_to(to=1000, range_vals=(1580, 1620))
    # spectrum.crop_data(bounds=(1620, 1750))
    # spectrum.baseline(method = "als", lam = 10**6, p=0.03, n_iter = 20)
    #
    # fig_2 = plotting_function(spectrum.data[spectrum.x_col], spectrum.data[spectrum.y_cols[0]],# label=name.split("_")[0],
    #                         fig=fig_2)
    spectra_2[name.split("_")[3]] =



fig_2.axes[0].vlines(1683, -20,100)
fig_2.axes[0].set_ylim(-20, 40)
plt.show()
print(spectra_2.keys())

fig_3 = None
spectra_3 = {}
for name, file in files_3.items():
    if "III" in name or "IV" in name:
        continue
    spectrum = Alpaca(logger=log)
    df = pd.read_csv(file, index_col=0 )
    # spectrum.load_data(file, from_filetype = "ramaberry")
    spectrum.load_data(df, from_filetype="ramaberry")

    spectrum.crop_data(bounds=(1000, 3500))
    spectrum.baseline(method="cholesky", lam=10 ** 6, p=0.03, n_iter=20)
    spectrum.normalise_to(to=1000, range_vals=(2900, 2970))
    spectrum.crop_data(bounds=(1520, 2000))
    # # spectrum.denoise(method = "savgol", window_frac = 0.01)
    # # spectrum.denoise(method="wavelet", level=1)
    spectrum.denoise(method = "median", window = 5)
    # # spectrum.denoise(method='fft', threshold=0.6)
    # # spectrum.normalise_to(to=1000, range_vals=(1580, 1620))
    spectrum.crop_data(bounds=(1620, 1750))
    spectrum.baseline(method = "als", lam = 10**6, p=0.03, n_iter = 20)

    fig_3 = plotting_function(spectrum.data[spectrum.x_col], spectrum.data[spectrum.y_cols[0]],# label=name.split("_")[0],
                            fig=fig_3)
    spectra_3[name.split("_")[1]] = spectrum


fig_3.axes[0].vlines(1683, -20,100)
fig_3.axes[0].set_ylim(-20, 40)
plt.show()


In [ ]:

integrals = pd.read_csv('error_estimates.csv')
integrals['Sample'] = ["run"+str(f) for f in integrals['Sample']]
integrals['conversion'] *= -1
print(integrals)

Now that worked quite well didn't it?

Now let's do something different, let's run the Vicuna algo and plot a different plot for each one to see how the data is modelled

In [ ]:
models = {}
for name, spectrum in spectra.items():
    print(f"Processing {name}")
    model = Vicuna(logger = log)
    model.absorb_class(class_data = spectrum)
    model.fit_all(lam = 1e6, p = 0.03, n_iter = 8, )
    models[name] = model
    # fig = plotting_function(spectrum.data[spectrum.x_col], model.data[spectrum.y_cols[0]], label=name)
    # plotting_function(model.model_spectrum[model.x_col],model.model_spectrum[model.y_cols[0]], label=f"{name}_model", fig=fig)
    for row in model.voigt_parameters[model.y_cols[0]].itertuples(index=True):
        if row.Index > 9:
            break
        A = row.A
        mu = row.mu
        sigma = row.sigma
        gamma = row.gamma

        # fig = plotting_function(model.data[model.x_col], model.voigt(model.data[model.x_col],A, mu, sigma, gamma), label=f"{mu:.1f}", fig=fig)



In [ ]:
from lamas.guanaco import Guanaco
import copy

from sklearn.linear_model import LinearRegression
import numpy as np
conc_dict_0 = {
    "H00D10.csv": {"Conc_H": 0.30, "Conc_D": 9.70},
    "H01D09.csv": {"Conc_H": 1.27, "Conc_D": 8.73},
    "H02D08.csv": {"Conc_H": 2.24, "Conc_D": 7.76},
    "H03D07.csv": {"Conc_H": 3.21, "Conc_D": 6.79},
    "H04D06.csv": {"Conc_H": 4.18, "Conc_D": 5.82},
    "H05D05.csv": {"Conc_H": 5.15, "Conc_D": 4.85},
    "H06D04.csv": {"Conc_H": 6.12, "Conc_D": 3.88},
    "H07D03.csv": {"Conc_H": 7.09, "Conc_D": 2.91},
    "H08D02.csv": {"Conc_H": 8.06, "Conc_D": 1.94},
    "H09D01.csv": {"Conc_H": 9.03, "Conc_D": 0.97},
    "H10D00.csv": {"Conc_H": 10.0, "Conc_D": 0.00}, }



conc_H = (250, 0)
conc_D = (10, 240)
conc_dict_temp = {name: {"Conc_H": conc_H[0]*(i/10)+conc_D[0]*(1-(i/10)), 'Conc_D':(1-(i/10))*conc_D[1]} for i, name in enumerate(conc_dict_0.keys())}
conc_dict_0 = copy.deepcopy(conc_dict_temp)


print(spectra_2.keys())
conc_dict_0 = {name: {'Conc_H': integrals.loc[integrals['Sample'] == name, 'conversion'].values[0], 
                      'Conc_D': integrals.loc[integrals['Sample'] == name, 'yield'].values[0]} 
               for name in spectra_2.keys() if name in integrals['Sample'].values}
print(conc_dict_0.keys())

conc_dict = {
"H250D000":{"Conc_H": 288, "Conc_D": 0},
"H200D050":{"Conc_H": 223, "Conc_D": 53},
"H175D075":{"Conc_H": 176, "Conc_D": 76},
"H125D125":{"Conc_H": 112, "Conc_D": 113},
"H100D150":{"Conc_H": 84,  "Conc_D": 165},
"H050D200":{"Conc_H": 46,  "Conc_D": 197},
"H015D235":{"Conc_H": 13,  "Conc_D": 212},
"ES125-I": {"Conc_H": 181, "Conc_D": 19},
"ES125-V": {"Conc_H": 56, "Conc_D": 157},
"ES125-VI": {"Conc_H": 24, "Conc_D": 174},
}

# ###
# clustering algo-not where we want it to be yet, pivot to linear regression:\


In [ ]:
area_dict = {}
mu_dict = {}
for name, model in models.items():

    peak_H = model.get_peak(y_col=model.y_cols[0], peak_position=1697, tolerance=9)
    peak_H_A = peak_H.area if peak_H is not None else 0
    peak_H_mu = peak_H.mu if peak_H is not None else 0
    peak_D = model.get_peak(y_col=model.y_cols[0], peak_position=1675, tolerance=5)
    peak_D_mu = peak_D.mu if peak_D is not None else 0
    peak_D_A = peak_D.area if peak_D is not None else 0

    area_dict[name] = {"H": peak_H_A, "D": peak_D_A}
    mu_dict[name] = {"H": peak_H_mu, "D": peak_D_mu}


# let's linear regress the values in conc_dict to the values in area_dict separately for H and D


regressor_H = LinearRegression()
regressor_D = LinearRegression()

y_H = np.array([area_dict[name]["H"] for name in conc_dict.keys()]).reshape(-1, 1)
X_H = np.array([conc_dict[name]["Conc_H"] for name in conc_dict.keys()]).reshape(-1, 1)

# enforce the 0,0 point and remove any other y_H = 0 rows:
X_H = X_H[y_H != 0].reshape(-1, 1)
y_H = y_H[y_H != 0].reshape(-1, 1)

y_D = np.array([area_dict[name]["D"] for name in conc_dict.keys()]).reshape(-1, 1)
X_D = np.array([conc_dict[name]["Conc_D"] for name in conc_dict.keys()]).reshape(-1, 1)

X_D = X_D[y_D != 0].reshape(-1, 1)
y_D = y_D[y_D != 0].reshape(-1, 1)

# enforce the fit to have 0,0 point


regressor_H.fit(y_H, X_H)
print(regressor_H.coef_)
print(regressor_H.intercept_)
print(regressor_H.score(X_H, y_H))
regressor_D.fit(y_D, X_D)
print(regressor_D.coef_)
print(regressor_D.intercept_)
print(regressor_D.score(X_D, y_D))

fig, ax = plt.subplots(1, 2, tight_layout=True)
ax[0].scatter(y_H, X_H)
ax[0].plot(y_H, regressor_H.predict(y_H), label = f"Linear fit score: {regressor_H.score(y_H, X_H):.2f}")
ax[0].set_title("Calibration H-Product")
ax[0].set_xlabel("Area 1687-1720 [cm$^{-1}$]")
ax[0].set_ylabel("Concentration H [mM]")
ax[1].scatter(y_D, X_D)
ax[1].plot(y_D, regressor_D.predict(y_D), label = f"Linear fit score: {regressor_D.score(y_D, X_D):.2f}")
ax[1].set_title("Calibration D-Product")
ax[1].set_xlabel("Area 1640-1687 [cm$^{-1}$]")
ax[1].set_ylabel("Concentration D [mM]")
ax[0].legend()
ax[1].legend()
plt.savefig('misperforming_2.png', dpi = 1000)
plt.show()

y_H_mu = np.array([mu_dict[name]["H"] for name in conc_dict.keys()]).reshape(-1, 1)
X_H_mu = np.array([conc_dict[name]["Conc_H"] for name in conc_dict.keys()]).reshape(-1, 1)

# enforce the 0,0 point and remove any other y_H = 0 rows:
X_H_mu = X_H_mu[y_H_mu != 0].reshape(-1, 1)
y_H_mu = y_H_mu[y_H_mu != 0].reshape(-1, 1)

y_D_mu = np.array([mu_dict[name]["D"] for name in conc_dict.keys()]).reshape(-1, 1)
X_D_mu = np.array([conc_dict[name]["Conc_D"] for name in conc_dict.keys()]).reshape(-1, 1)

# enforce the fit to have 0,0 point
X_D_mu = X_D_mu[y_D_mu != 0].reshape(-1, 1)
y_D_mu = y_D_mu[y_D_mu != 0].reshape(-1, 1)


regressor_H_mu = LinearRegression()
regressor_D_mu = LinearRegression()

regressor_H_mu.fit(X_H_mu, y_H_mu)
print(regressor_H_mu.coef_)
print(regressor_H_mu.intercept_)
print(regressor_H_mu.score(X_H_mu, y_H_mu))
regressor_D_mu.fit(X_D_mu, y_D_mu)
print(regressor_D_mu.coef_)
print(regressor_D_mu.intercept_)
print(regressor_D_mu.score(X_D_mu, y_D_mu))


fig, ax = plt.subplots(1, 2, tight_layout=True)
ax[0].scatter(y_H_mu, X_H_mu)
# ax[0].plot(X_H_mu, regressor_H.predict(X_H_mu))
ax[0].set_title("Calibration H-Product")
ax[0].set_xlabel("Area 1687-1720 [cm$^{-1}$]")
ax[0].set_ylabel("Concentration H [mM]")
ax[0].set_ylim(1680, 1700)
ax[1].scatter(X_D_mu, y_D_mu)
# ax[1].plot(X_D_mu, regressor_D.predict(X_D_mu))
ax[1].set_ylim(1660, 1680)
ax[1].set_title("Calibration D-Product")
ax[1].set_xlabel("Area 1640-1687 [cm$^{-1}$]")
ax[1].set_ylabel("Concentration D [mM]")
# plt.savefig('misperforming.png', dpi = 1000)
plt.show()








In [ ]:
# now insanity check:
# lets see if we integrate from the split point left and right if we get better linear regressors?

isosbestic_point = 1683
isosbestic_point_shifted = 1681
D = (1675, isosbestic_point)
H = (isosbestic_point, 1715)
D2 = (1675, isosbestic_point_shifted)
H2 = (isosbestic_point_shifted, 1715)

area_integrals = {}
for name, model in spectra.items():
    area_integrals[name] = {"H": model._integrate_basic(bounds = H), "D": model._integrate_basic(bounds = D)}


area_integrals_tests = {}
for name,spectrum in spectra_2.items():
    if "isotope" in name:
        continue
    area_integrals_tests[name] = {"H": spectrum._integrate_basic(bounds = H2), "D": spectrum._integrate_basic(bounds = D2)}

area_integrals_3 = {}
for name, model in spectra_3.items():
    area_integrals_3[name] = {"H": model._integrate_basic(bounds = H2), "D": model._integrate_basic(bounds = D2)}

print(area_integrals)
print(area_integrals_tests)
print(area_integrals_3)


In [ ]:
# make linear models and plot:
regressor_H_integral = LinearRegression()
regressor_D_integral = LinearRegression()

X_H_integral = np.array([area_integrals[name]["H"] for name in conc_dict.keys()]).reshape(-1, 1)
y_H_integral = np.array([conc_dict[name]["Conc_H"] for name in conc_dict.keys()]).reshape(-1, 1)
X_H_test = np.array([area_integrals_tests[name]["H"] for name in conc_dict_0.keys()]).reshape(-1, 1)
y_H_test = np.array([conc_dict_0[name]["Conc_H"] for name in conc_dict_0.keys()]).reshape(-1, 1)
print(X_H_test- y_H_test)
#
X_D_integral = np.array([area_integrals[name]["D"] for name in conc_dict.keys()]).reshape(-1, 1)
y_D_integral = np.array([conc_dict[name]["Conc_D"] for name in conc_dict.keys()]).reshape(-1, 1)
X_D_test = np.array([area_integrals_tests[name]["D"] for name in conc_dict_0.keys()]).reshape(-1, 1)
y_D_test = np.array([conc_dict_0[name]["Conc_D"] for name in conc_dict_0.keys()]).reshape(-1, 1)
print(X_D_test- y_D_test)


In [ ]:
integrals = pd.read_csv('integral.csv')
test = pd.read_csv('test.csv')

X_H_integral = integrals['X_H'].values.reshape(-1, 1)
y_H_integral = integrals['y_H'].values.reshape(-1, 1)
X_D_integral = integrals['X_D'].values.reshape(-1, 1)
y_D_integral = integrals['y_D'].values.reshape(-1, 1)
X_H_test = test['X_H'].values.reshape(-1, 1)
y_H_test = test['y_H'].values.reshape(-1, 1)
X_D_test = test['X_D'].values.reshape(-1, 1)
y_D_test = test['y_D'].values.reshape(-1, 1)

In [ ]:

regressor_H_integral = LinearRegression()
regressor_D_integral = LinearRegression()

regressor_H_integral.fit(X_H_integral, y_H_integral)
print('coeff')
print(regressor_H_integral.coef_)
print(regressor_H_integral.intercept_)
print('score_self')
print(regressor_H_integral.score(X_H_integral, y_H_integral))
print('score_test')
print(regressor_H_integral.score(X_H_test, y_H_test))

regressor_D_integral.fit(X_D_integral, y_D_integral)
print('coeff')
print(regressor_D_integral.coef_)
print(regressor_D_integral.intercept_)
print('score_self')
print(regressor_D_integral.score(X_D_integral, y_D_integral))
print('score_test')
print(regressor_D_integral.score(X_D_test, y_D_test))

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict

def approach_3_cv_residuals_with_bins(X, y, compound_name='Compound', n_splits=5, n_bins=5):
    """
    1) K-fold cross-validation: integral = a + b * concentration.
    2) Calculate out-of-sample residuals.
    3) Bin integrals -> get (bin_means, bin_stds).
    4) Return (bin_means, bin_stds) for interpolation.
    """
    color = '#fadeb9' if compound_name == 'H' else '#8b97cc'
    prettyname = '4-Bromobenzaldehyde' if compound_name == 'H' else '4-Bromobenzaldehyde-D'
    integral_range = '1681-1715 [cm$^{-1}$]' if compound_name == 'H' else '1675-1681 [cm$^{-1}$]'
    X = X.reshape(-1, 1)  # concentration
    y = y.reshape(-1, 1)  # integral

    # Cross-val predictions
    model = LinearRegression()
    y_pred_cv = cross_val_predict(model, X, y, cv=n_splits).flatten()

    # Residuals
    y_actual = y.flatten()
    residuals_cv = y_actual - y_pred_cv

    # Plot cross-validated residuals vs. actual integral
    plt.figure(figsize=(4, 4), tight_layout=True)
    plt.scatter(y_actual, residuals_cv, color = color)
    plt.axhline(y=0, color='r', linestyle='--', label='Zero residual')
    plt.xlabel(f'Integral: {integral_range}')
    plt.ylabel(f'Residual {compound_name} (Measured - CV Prediction)')
    # plt.title(f'Cross-Validated Residuals vs. Integral ({compound_name})')
    plt.legend()
    plt.savefig(f'cv_residuals_{compound_name}.svg',format = 'svg',dpi=300)
    plt.show()

    # Sort by integral, bin integrals
    sorted_idx = np.argsort(y_actual)
    y_sorted = y_actual[sorted_idx]
    res_sorted = residuals_cv[sorted_idx]

    bins = np.array_split(range(len(y_sorted)), n_bins)
    bin_means = []
    bin_stds = []
    for b in bins:
        bin_means.append(np.mean(y_sorted[b]))
        bin_stds.append(np.std(res_sorted[b]))

    bin_means = np.array(bin_means)
    bin_stds = np.array(bin_stds)

    # Plot bin-wise std dev vs. integral
    plt.figure(figsize=(4, 4), constrained_layout=True)
    plt.plot(bin_means, bin_stds, 'o--', label='Bin-wise Std Dev', color = color)
    plt.xlabel(f'Integral: {integral_range}')
    plt.ylabel(f'$\sigma$ (Out-of-Sample Error) {prettyname}')
    # plt.title(f'Out-of-Sample Error vs. Integral ({compound_name})')
    plt.legend()
    plt.savefig(f'cv_sigmas_{compound_name}.svg',format = 'svg',dpi=300)
    plt.show()

    # Print overall CV error
    cv_std = np.std(residuals_cv)
    print(f'[{compound_name}] Overall out-of-sample measurement error (std) ~ {cv_std:.4f}')

    # Return bin data for interpolation
    return bin_means, bin_stds

def interpolate_std_for_new_integral(
    I_new: float,
    bin_means: np.ndarray,
    bin_stds: np.ndarray,
) -> float:
    """
    Given a new integral value I_new,
    and arrays of (bin_means, bin_stds) that define a piecewise
    (or binned) measurement error model,
    return an interpolated (or extrapolated) standard deviation.

    - If I_new is within the bin range, do piecewise linear interpolation.
    - If I_new is below the first bin_mean, extrapolate from the first two bins.
    - If I_new is above the last bin_mean, extrapolate from the last two bins.
    """
    # If the range is trivial or there's only one bin, just return the single std
    if len(bin_means) < 2:
        return float(bin_stds[0])

    # Case 1: extrapolate below
    if I_new < bin_means[0]:
        x1, x2 = bin_means[0], bin_means[1]
        y1, y2 = bin_stds[0], bin_stds[1]
        slope = (y2 - y1) / (x2 - x1)
        return y1 + slope * (I_new - x1)

    # Case 2: extrapolate above
    if I_new > bin_means[-1]:
        x1, x2 = bin_means[-2], bin_means[-1]
        y1, y2 = bin_stds[-2], bin_stds[-1]
        slope = (y2 - y1) / (x2 - x1)
        return y2 + slope * (I_new - x2)

    # Case 3: within bin range, do piecewise interpolation
    for i in range(len(bin_means) - 1):
        if bin_means[i] <= I_new <= bin_means[i+1]:
            x1, x2 = bin_means[i], bin_means[i+1]
            y1, y2 = bin_stds[i], bin_stds[i+1]
            slope = (y2 - y1) / (x2 - x1)
            return y1 + slope * (I_new - x1)

    # Fallback (should not happen if bins cover entire range)
    return float(bin_stds[-1])

#0) prep_data 
bins_D, stds_D = approach_3_cv_residuals_with_bins(X_D_integral, y_D_integral, compound_name='D')
bins_H, stds_H = approach_3_cv_residuals_with_bins(X_H_integral, y_H_integral, compound_name='H')

# 1) FLATTEN your test arrays (so they're 1D, not Nx1)
y_H_test = y_H_test.flatten()
y_D_test = y_D_test.flatten()

# 2) Build error arrays as 1D lists of floats
D_errors_test = []
for I_new in y_D_test:
    std_estimate = interpolate_std_for_new_integral(I_new, bins_D, stds_D)
    D_errors_test.append(std_estimate)   # single float

H_errors_test = []
for I_new in y_H_test:
    std_estimate = interpolate_std_for_new_integral(I_new, bins_H, stds_H)
    H_errors_test.append(std_estimate)   # single float

D_errors_test = np.array(D_errors_test)  # make them arrays if you prefer
H_errors_test = np.array(H_errors_test)


In [ ]:
from joblib import dump, load


model_dump = {
    "model_SM": regressor_H_integral,
    "model_PI": regressor_D_integral,
    "X_SM": X_H_integral,
    "y_SM": y_H_integral,
    "X_PI": X_D_integral,
    "y_PI": y_D_integral,
}
dump(model_dump, "./integral_models.linear_model")

In [ ]:
model = load('./integral_models.linear_model')
model
# print(model)
# y_SM = model['y_SM']
# X_SM = model['X_SM']
# for i, item in enumerate(X_SM):
#     print(X_SM, y_SM[i])
# model.predict([[170]])